# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://diploi.com")
links

['/',
 '/',
 '/features/dev',
 '/features/teams',
 '/features/devops',
 '/features/ai',
 '/features/dev#editors',
 '/features/devops#deploy',
 '/features/devops#iac',
 '/features/dev#stack-builder',
 '/features/devops#deploy',
 '/features/dev#deploy',
 '/features/devops#cdn',
 '/features/teams#security',
 '/import',
 '/lovable',
 '/features/dev',
 '/features/teams',
 '/features/devops',
 '/features/ai',
 '/pricing',
 '/blog',
 'https://docs.diploi.com/',
 'https://console.diploi.com',
 'https://calendly.com/hello-diploi/30min',
 'https://github.com/diploi',
 'https://discord.gg/vvgQxVjC8G',
 '/features/dev',
 '/features/teams',
 '/features/devops',
 '/features/ai',
 'mailto:hello@diploi.com',
 'https://console.diploi.com/sign-up',
 'mailto:hello@diploi.com',
 'https://calendly.com/hello-diploi/30min',
 '/launchweek',
 '/pricing',
 '/blog',
 '/jobs',
 'https://docs.diploi.com/',
 'https://docs.diploi.com/reference/glossary',
 '/faq',
 'https://console.diploi.com/login',
 '/alternative/t

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [10]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://diploi.com")

{'links': [{'type': 'about page', 'url': 'https://diploi.com/about'},
  {'type': 'careers page', 'url': 'https://diploi.com/jobs'},
  {'type': 'pricing page', 'url': 'https://diploi.com/pricing'},
  {'type': 'blog page', 'url': 'https://diploi.com/blog'},
  {'type': 'documentation page', 'url': 'https://docs.diploi.com/'},
  {'type': 'documentation page',
   'url': 'https://docs.diploi.com/reference/glossary'},
  {'type': 'GitHub page', 'url': 'https://github.com/diploi'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/diploidev/'},
  {'type': 'dev.to page', 'url': 'https://dev.to/diploi'},
  {'type': 'YouTube channel', 'url': 'https://www.youtube.com/@diploi'},
  {'type': 'Bluesky profile', 'url': 'https://bsky.app/profile/diploi.com'},
  {'type': 'Discord server', 'url': 'https://discord.gg/vvgQxVjC8G'},
  {'type': 'Launchweek page', 'url': 'https://diploi.com/launchweek'},
  {'type': 'FAQ page', 'url': 'https://diploi.com/faq'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},  
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
baidu/Unlimited-OCR
Updated
3 days ago
•
2.59M
•
3.15k
poolside/Laguna-S-2.1
Updated
2 days ago
•
56.4k
•
679
upstage/Solar-Open2-250B
Updated
2 days ago
•
3.31k
•
582
DavidAU/Qwen3.6-27B-Fable-Fusion-711-Uncensored-Heretic-NM-DAU-NEO-MAX-MTP-GGUF
Updated
about 12 hours ago
•
552k
•
588
Nanbeige/Nanbeige4.2-3B
Updated
about 2 hours ago
•
14k
•
425
Browse 2M+ models
Spaces
Running


In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nbaidu/Unlimited-OCR\nUpdated\n3 days ago\n•\n2.59M\n•\n3.15k\npoolside/Laguna-S-2.1\nUpdated\n2 days ago\n•\n56.4k\n

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face - The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the leading platform and community driving the future of Artificial Intelligence. It is the home where machine learning researchers, developers, and enterprises come together to collaborate on cutting-edge AI models, datasets, and applications.

With a vibrant open ecosystem, Hugging Face empowers AI innovation at scale, supporting over **2 million models**, **500,000+ datasets**, and **1 million+ applications** available to explore and utilize.

---

## Platform Highlights

- **Models**: Access and contribute to a vast repository of state-of-the-art AI models including natural language processing, computer vision, and more.
- **Datasets**: Explore and share a comprehensive collection of datasets for training and benchmarking AI systems.
- **Spaces**: Deploy interactive machine learning apps that run on Hugging Face's infrastructure, fostering collaboration and demonstration.
- **HuggingChat**: Cutting-edge AI chat tools powered by community-driven innovations.
- **Enterprise Solutions**: Tailored plans for teams and organizations, including enterprise-grade support, inference endpoints, storage, and advanced workload management.

---

## Community & Culture

At its core, Hugging Face is a **community-first organization** that emphasizes openness, collaboration, and knowledge sharing. This culture fosters:

- **Global Collaboration**: Developers and researchers worldwide contribute models, datasets, and tools openly.
- **Continuous Learning**: Regular blog posts, papers, forums, and a vibrant Discord channel enable ongoing education and discussion.
- **Innovation-Driven**: Open forums including GitHub repositories and daily papers keep the community at the forefront of AI advances.
- **Inclusive & Supportive Environment**: Whether you’re a hobbyist, researcher, or enterprise, Hugging Face offers resources and spaces to grow and succeed.

---

## Customers & Use Cases

Hugging Face serves a broad range of users including:

- **Academics & Researchers** building and sharing state-of-the-art AI models.
- **Enterprises & Teams** leveraging AI for product development, custom solutions, and scalable deployment.
- **Developers & Hobbyists** accessing pre-trained models and participating in collaborative AI projects.
- **Innovative Startups** integrating AI to transform industries and user experiences.

Popular uses include text understanding, image and video AI applications, automated OCR, large language model fine-tuning, and real-time AI inferencing.

---

## Careers at Hugging Face

Joining Hugging Face means becoming part of a **mission-driven team** dedicated to advancing open AI technology globally. The company values curiosity, collaboration, and impact, offering roles in:

- Machine Learning Research & Engineering
- Software Development & Infrastructure
- Developer Relations & Community Management
- Enterprise Solutions & Customer Success
- Product & Design

Interested candidates can explore current openings and apply through their [website](https://huggingface.co).

---

## Why Choose Hugging Face?

- Access to an unparalleled AI ecosystem and community.
- State-of-the-art tools and APIs for AI model development and deployment.
- Enterprise-ready solutions with dedicated support.
- A thriving collaborative culture focused on open science and innovation.
- Opportunities to learn, contribute, and lead in the fast-evolving AI landscape.

---

Explore the future of AI with Hugging Face – where innovation is collaborative, accessible, and built together.

Visit [huggingface.co](https://huggingface.co) to start your journey.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. Positioned as the collaboration platform for the machine learning community, Hugging Face empowers ML engineers, scientists, and end users to learn, explore, share, and experiment with open-source machine learning models, datasets, and applications.

At the heart of an open and ethical AI future, Hugging Face supports a fast-growing community, hosts over 2 million models, and offers a rich ecosystem of tools, including Spaces, Buckets, and HuggingChat. The company is committed to advancing AI through openness, collaboration, and innovation.

---

## What Hugging Face Offers

- **Hugging Face Hub**: The central repository to discover and share ML models and datasets — with over 2 million models and 500,000+ datasets available.
- **Spaces**: Collaborative environments to run AI applications and demos, such as image-to-video generators and OCR tools.
- **Enterprise Solutions**: Tailored team and enterprise plans including Hugging Face PRO, enterprise support, inference endpoints, inference providers, and storage buckets to scale AI across businesses.
- **Community & Learning**: Active forums, Discord channels, daily research paper postings, blogs, and educational resources empowering users worldwide.
- **Open-source tools**: Leading open-source libraries and toolkits that are widely used by researchers and industry professionals.

---

## Company Culture & Values

- **Community-driven**: Hugging Face thrives as an open platform where collaboration, sharing, and ethical AI development are core.
- **Innovation-focused**: Constantly pushing the boundaries with a talented science team exploring the cutting edge of AI and machine learning technologies.
- **Accessible AI**: Dedicated to lowering barriers to entry, enabling the next generation of researchers and developers with tools and resources.
- **Transparency & Ethics**: Committed to building a responsible AI future through openness and shared resources.

---

## Customers & Partners

Hugging Face supports a diverse range of users and organizations including:

- Independent machine learning researchers and developers.
- Academic institutions and scientific communities.
- Large enterprises adopting AI at scale via the Team & Enterprise plans.
- Open-source contributors and enthusiasts worldwide.

Their platform powers applications across industries by making advanced machine learning models easily accessible and deployable.

---

## Careers at Hugging Face

Are you passionate about AI and open collaboration? Hugging Face offers exciting career opportunities in:

- Machine Learning Research and Engineering
- Software Development
- Community Management
- Product and Enterprise Solutions

Join a vibrant global team dedicated to shaping the future of AI through innovation and openness.

Visit the Hugging Face Careers page to explore open positions and become part of the AI revolution.

---

## Connect with Hugging Face

- Website: https://huggingface.co
- GitHub: https://github.com/huggingface
- Twitter: https://twitter.com/huggingface
- LinkedIn: https://linkedin.com/company/huggingface
- Community Forum & Discord for discussions and support

---

## Brand Identity

**Colors:**  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

**Logo formats:** Available in SVG, PNG, AI for use in projects and branding.

---

**Hugging Face is leading the AI revolution by bringing together an open, collaborative community to build the future of machine learning. Join them and be part of the change!**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>